# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadbinijaz17/flyrankAI_Intern_ML/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I chose **Lane 2: Refresh / Content Opportunity Scoring**. I've learned ML concepts before — things like feature engineering, training models, and evaluating them — but I never really applied them to a real-world problem from start to finish. This lane gives me that chance. The starter pipeline already has working code for a baseline score, logistic regression, decision tree, and random forest, so I can actually compare my work against something real instead of guessing if I'm doing it right. I want to see if I can build something that actually helps someone decide which page to fix first. That feels more meaningful than just getting a high accuracy number.

In [9]:
# Supporting check: count pages per trend direction to understand the label balance
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(df['trend_direction'].value_counts())
print(f"\nDeclining pages: {df['trend_direction'].value_counts().get('down', 0)} out of {len(df)}")


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining pages: 16262 out of 30000


## 2. The question: decision, action, cost of a wrong call

An editor or SEO person has limited time. They can't review all 30,000 pages to decide which ones need a refresh. My work helps them answer one question: **which page should I look at first?**

- **Decision:** Which page to review next for a content refresh.
- **Who acts:** A content editor or SEO specialist who reviews the top-ranked pages.
- **Wrong call cost:** If my model says "fix this page" but it's actually fine, the editor wasted their time. If it misses a page that's really declining fast, that page keeps losing traffic and nobody notices.
- **Why ML helps:** There are too many signals to track by hand — impressions, clicks, CTR, position, content age, engagement rates, scroll depth. They all interact in messy ways. A simple rule like "refresh old pages" misses the pages that are old but still performing well, or young but already dropping. ML can find patterns a human wouldn't spot.

In [10]:
# Quick check: how many pages have enough data to be worth reviewing?
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
visible_pages = df[(df['impressions_90d'] >= 500) & (df['sessions_90d'] >= 10)]
print(f"Pages with enough visibility to review: {len(visible_pages)} out of {len(df)}")
print(f"That's {len(visible_pages)/len(df)*100:.1f}% of the dataset")


Pages with enough visibility to review: 11696 out of 30000
That's 39.0% of the dataset


## 3. Quick look at the data (2-3 real numbers)

Loading the starter CSV gave me some numbers that convinced me this lane is worth the next 7 weeks:

- **54.2% of pages are already declining**  that's over half the dataset with `trend_direction == "down"`. This isn't a rare problem, it's everywhere.
- **The baseline already gets 0.627 ROC AUC** and the random forest gets **0.750 ROC AUC with 0.740 Precision@50**. That means the starter model gets about 37 out of the top 50 pages right. That's good but not great there's room to improve.
- **1,205 pages have avg_position = 0** meaning no position data at all. That's a data quality issue I'll need to handle.

These numbers tell me there's a real problem (lots of declining pages), a working baseline to beat, and clear data challenges to solve.

In [11]:
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

total = len(df)
declining = (df['trend_direction'] == 'down').sum()
no_position = (df['avg_position'] == 0).sum()
avg_impressions = df['impressions_90d'].mean()

print(f"Total pages: {total}")
print(f"Declining pages: {declining} ({declining/total*100:.1f}%)")
print(f"Pages with no position data: {no_position} ({no_position/total*100:.1f}%)")
print(f"Average 90-day impressions: {avg_impressions:.0f}")


Total pages: 30000
Declining pages: 16262 (54.2%)
Pages with no position data: 1205 (4.0%)
Average 90-day impressions: 5200


## 4. Careful words: what I can and can't claim

**What I CAN claim:**
- I observed that certain signals (like low CTR for a page's position tier, or high impressions with declining trend) are associated with pages that need review.
- My model ranks pages by priority — it's a decision-support tool, not a guarantee.
- The recommendations are directional: an editor should look at the top-ranked pages first, but use their own judgment.
- I can say "pages with this pattern were more likely to be declining in our dataset" — not "this pattern causes decline."

**What I CANNOT claim:**
- I cannot claim that refreshing a page will definitely recover its traffic. That would need a controlled experiment.
- I cannot claim I proved how Google's algorithm works. I only observed search signals, not Google's internal logic.
- I cannot claim AI citations or rankings — the data only measures click-throughs from AI tools, not whether AI platforms cite or rank the content.
- I cannot make causal claims. This is observational analysis, not a randomized experiment.

In [12]:
# Check: what percentage of pages have AI sessions? (to understand sparsity)
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
has_ai = (df['ai_sessions_90d'] > 0).sum()
print(f"Pages with AI sessions: {has_ai} out of {len(df)} ({has_ai/len(df)*100:.2f}%)")
print(f"This confirms AI data is sparse - I won't build a binary classifier on it alone.")


Pages with AI sessions: 1930 out of 30000 (6.43%)
This confirms AI data is sparse - I won't build a binary classifier on it alone.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.